In [2]:
from pathlib import Path
import json
import polars as pl

# File locations (same folder as this notebook)
base_dir = Path.cwd()
input_csv = base_dir / "large.csv"
mapping_json = base_dir / "security_mapping.json"
output_csv = base_dir / "EGX30-2025-2.csv"

# Validate required input files
if not input_csv.exists():
    raise FileNotFoundError(f"Input file not found: {input_csv}")
if not mapping_json.exists():
    raise FileNotFoundError(f"Mapping file not found: {mapping_json}")

# Load security-code-to-symbol mapping
with mapping_json.open("r", encoding="utf-8") as f:
    mapping = json.load(f)

# Read source data
df = pl.read_csv(
    input_csv,
    separator=";",
    has_header=False,
    new_columns=[
        "date",
        "security_code",
        "type",
        "transaction_id",
        "price",
        "volume",
        "open",
        "close",
        "datetime",
    ],
)

# Normalize security_code and map symbols
mapping_df = pl.DataFrame({
    "security_code": list(mapping.keys()),
    "security_symbol": list(mapping.values()),
})

df = (
    df.with_columns(
        pl.col("security_code").str.replace_all('"', "")
    )
    .join(mapping_df, on="security_code", how="left")
)

cols = df.columns
cols.remove("security_symbol")

idx = cols.index("security_code") + 1
cols = cols[:idx] + ["security_symbol"] + cols[idx:]

df = df.select(cols)

# Rename columns
df = df.rename({
    "date" : "TRADE_DATE",
    "volume": "VOLUME_TRADED",
    "open": "TRADE_VAL",
    "type": "MARKET_CODE",
    "datetime": "EXECUTION_TIME",
    "security_code": "SYMBOL_CODE",
"security_symbol": "SECURITY_NAME",
"transaction_id": "TICKET_ID",
"price": "TRADE_PRICE",

})

# drop duplicate column
df = df.drop("close")
df = df.drop("MARKET_CODE")

# Write output
df.write_csv(output_csv)
print(f"Done. Output saved to: {output_csv}")
print(f"Rows: {df.height}, Columns: {df.width}")
df.head()

Done. Output saved to: e:\CSE\CSE4-1\Graduation project\Data\MD - Ebrahim Alaa El-Din\EGX30-2025-2.csv
Rows: 8032899, Columns: 8


TRADE_DATE,SYMBOL_CODE,SECURITY_NAME,TICKET_ID,TRADE_PRICE,VOLUME_TRADED,TRADE_VAL,EXECUTION_TIME
str,str,str,str,f64,i64,f64,str
"""02/01/2025""","""EGS30901C010""","""Juhayna Food Industries""","""20250102-000200223036""",33.49,1,33.49,"""02/01/2025 10:00:00 AM"""
"""02/01/2025""","""EGS30901C010""","""Juhayna Food Industries""","""20250102-000200223037""",33.49,3,100.47,"""02/01/2025 10:00:00 AM"""
"""02/01/2025""","""EGS30901C010""","""Juhayna Food Industries""","""20250102-000200223038""",33.49,127,4253.23,"""02/01/2025 10:00:00 AM"""
"""02/01/2025""","""EGS30901C010""","""Juhayna Food Industries""","""20250102-000200223039""",33.49,43,1440.07,"""02/01/2025 10:00:00 AM"""
"""02/01/2025""","""EGS30901C010""","""Juhayna Food Industries""","""20250102-000200223040""",33.49,136,4554.64,"""02/01/2025 10:00:00 AM"""
